### 1. Setup and Installations

Installs all required libraries and sets global configuration variables for the RAG pipeline.
**LLM:** OpenAI `gpt-4o-mini` — replaces the original `distilgpt2` for accurate, coherent answers.

**Important:** To use the OpenAI API, the system first tries to read your key from `/content/drive/MyDrive/Upgrad_GenAI/HelpmateAI_dataset/myopenai_key.txt`. If that file is not found or cannot be read, it will then check for an `OPENAI_API_KEY` set as a Colab secret. To use Colab secrets, click the "🔑" icon in the left panel, then "Add a new secret", name it `OPENAI_API_KEY`, and paste your key there. The code automatically picks it up.

In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
# Install dependencies
!pip install -q chromadb rfc3987 transformers PyPDF2 sentence-transformers langchain-text-splitters pdf2image pytesseract openai
!pip install -q --upgrade opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc
!sudo apt-get update -qq && sudo apt-get install -y -qq poppler-utils tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [42]:
import os
import re
import pandas as pd
import PyPDF2
import pytesseract
from pdf2image import convert_from_path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# --- Global Configuration ---
DATA_DIRECTORY       = '/content/drive/MyDrive/Upgrad_GenAI/HelpmateAI_dataset'
CHROMADB_PATH        = '/content/drive/MyDrive/Upgrad_GenAI/HelpmateAI_dataset/chromadb_data_v2'
COLLECTION_NAME      = 'medical_documents_rag'
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
OPENAI_MODEL         = 'gpt-4o-mini'   # cheap, fast, instruction-tuned

# --- OpenAI client ---
# Priority: 1. From specified file; 2. From Colab secret (environment variable)
OPENAI_API_KEY_FILE = '/content/drive/MyDrive/Upgrad_GenAI/HelpmateAI_dataset/myopenai_key.txt'
OPENAI_API_KEY = None

if os.path.exists(OPENAI_API_KEY_FILE):
    try:
        with open(OPENAI_API_KEY_FILE, 'r') as f:
            OPENAI_API_KEY = f.read().strip()
        print(f"OpenAI API key loaded from '{OPENAI_API_KEY_FILE}'.")
    except Exception as e:
        print(f"Error reading API key from file: {e}. Falling back to Colab secrets.")

if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    if OPENAI_API_KEY:
        print("OpenAI API key loaded from Colab secrets.")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Please set it in the specified file or as a Colab secret.")

openai_client  = OpenAI(api_key=OPENAI_API_KEY)

print("Libraries imported and global configurations set.")

# Load CrossEncoder re-ranker (still local/free)
print("Loading CrossEncoder model for re-ranking...")
re_ranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("CrossEncoder model loaded.")

OpenAI API key loaded from '/content/drive/MyDrive/Upgrad_GenAI/HelpmateAI_dataset/myopenai_key.txt'.
Libraries imported and global configurations set.
Loading CrossEncoder model for re-ranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CrossEncoder model loaded.


### 2. Core RAG Functions

Modular functions for the full RAG pipeline:
- `load_and_process_pdfs` — extracts text (PyPDF2 → OCR fallback)
- `chunk_documents` — splits text into overlapping chunks
- `generate_embeddings` — encodes chunks with a SentenceTransformer
- `setup_chromadb` — initialises and populates the vector store
- `retrieve_documents` — semantic search against ChromaDB
- `re_rank_documents` — cross-encoder re-ranking of retrieved results
- `generate_answer` — **OpenAI gpt-4o-mini** answer synthesis from context

In [43]:
def load_and_process_pdfs(directory):
    """
    Load PDFs from a directory, extracting text page by page. Tries PyPDF2 first;
    falls back to OCR via pytesseract if no text is extracted for a page.  Returns
    a DataFrame with columns ['doc_id', 'page_number', 'text'].
    """
    if not os.path.isdir(directory):
        print(f"Error: Directory not found at '{directory}'.")
        return pd.DataFrame()

    all_page_records = []
    for filename in os.listdir(directory):
        if not filename.endswith('.pdf'):
            continue

        filepath = os.path.join(directory, filename)
        file_page_records = []
        try:
            # Attempt text extraction with PyPDF2 page by page
            with open(filepath, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                num_pages = len(reader.pages)
                for i, page in enumerate(reader.pages):
                    page_text = page.extract_text() or ""
                    if page_text.strip():
                        file_page_records.append({'doc_id': filename, 'page_number': i + 1, 'text': page_text.strip()})

                # Fall back to OCR if PyPDF2 yielded no text for any page or if total text is empty
                if not file_page_records or (sum(len(rec['text']) for rec in file_page_records) == 0):
                    print(f"No text from PyPDF2 for '{filename}' — trying OCR page by page...")
                    # Clear PyPDF2 records if they were empty or contained only whitespace
                    file_page_records = []
                    images = convert_from_path(filepath, dpi=300)
                    for i, image in enumerate(images):
                        ocr_text = pytesseract.image_to_string(image)
                        if ocr_text.strip():
                            file_page_records.append({'doc_id': filename, 'page_number': i + 1, 'text': ocr_text.strip()})

                    if file_page_records:
                        print(f"  OCR succeeded for '{filename}'. {len(file_page_records)} pages extracted.")
                    else:
                        print(f"  OCR also failed for '{filename}'.")
                else:
                    print(f"PyPDF2 extracted text from '{len(file_page_records)} pages of '{filename}'.")

            all_page_records.extend(file_page_records)

        except Exception as exc:
            print(f"Error processing '{filename}': {exc}")

    if not all_page_records:
        print(f"No text extracted from any PDF in '{directory}'.")
        return pd.DataFrame()

    print(f"\nLoaded {len(all_page_records)} pages from '{len(os.listdir(directory))} PDF(s)' in '{directory}'.")
    return pd.DataFrame(all_page_records)

In [44]:
def chunk_documents(df_documents):
    """
    This function now treats each row (which represents a page) as a chunk.
    It renames columns for consistency with later RAG steps.
    Returns a DataFrame with columns ['text', 'source_document', 'page_number'].
    """
    # Each row in df_documents is already a page, so it's effectively a chunk.
    df_chunks = df_documents.copy()

    # Ensure the required columns for later steps are present
    df_chunks = df_chunks.rename(columns={'doc_id': 'source_document'}) # Align with ChromaDB metadata

    print(f"Treated {len(df_chunks)} pages as chunks.")
    return df_chunks

In [45]:
def generate_embeddings(df_chunks, model_name):
    """
    Encode each chunk with a SentenceTransformer model.
    Adds an 'embedding' column to df_chunks (in-place copy).
    Returns (df_chunks_with_embeddings, embedding_model).
    """
    print(f"Initialising embedding model: {model_name}...")
    model = SentenceTransformer(model_name)

    # Encode all texts in a single batch — much faster than row-by-row apply()
    print("Generating embeddings...")
    embeddings = model.encode(df_chunks['text'].tolist(), show_progress_bar=True)
    df_out = df_chunks.copy()
    df_out['embedding'] = embeddings.tolist()

    print(f"Done — {len(df_out)} embeddings, {len(df_out['embedding'].iloc[0])} dimensions each.")
    return df_out, model

In [46]:
def setup_chromadb(df_chunks_with_embeddings, client_path, collection_name, embedding_model_name):
    """
    Create (or recreate) a ChromaDB persistent collection and bulk-add
    all chunks.  Returns (client, collection).
    """
    print(f"Initialising ChromaDB at: {client_path}")
    client = chromadb.PersistentClient(path=client_path)
    embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=embedding_model_name
    )

    # Drop existing collection for a clean rebuild
    try:
        client.delete_collection(name=collection_name)
        print(f"Deleted existing collection '{collection_name}'.")
    except Exception:
        print(f"No existing collection '{collection_name}' to delete.")

    collection = client.get_or_create_collection(
        name=collection_name, embedding_function=embed_fn
    )

    if collection.count() == 0:
        n = len(df_chunks_with_embeddings)
        print(f"Adding {n} documents to '{collection_name}'...")
        # Use 'source_document' and 'page_number' as metadata
        collection.add(
            ids=[f"{row['source_document']}_page_{row['page_number']}" for idx, row in df_chunks_with_embeddings.iterrows()],
            embeddings=df_chunks_with_embeddings['embedding'].tolist(),
            documents=df_chunks_with_embeddings['text'].tolist(),
            metadatas=df_chunks_with_embeddings[['source_document', 'page_number']]
                       .to_dict('records'),
        )
        print(f"Collection populated with {collection.count()} documents.")
    else:
        print(f"Collection already has {collection.count()} documents — skipping add.")

    return client, collection

In [47]:
def retrieve_documents(query_text, client_path, collection_name, embedding_model_name,
                       n_results=10, document_filter=None):
    """
    Query ChromaDB for the top-n_results chunks closest to query_text.
    Optionally filter by source document name.  Returns a ChromaDB
    results dict, or None on error.
    """
    try:
        client = chromadb.PersistentClient(path=client_path)
        embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=embedding_model_name
        )
        collection = client.get_collection(
            name=collection_name, embedding_function=embed_fn
        )
        print(f"Connected to '{collection_name}' ({collection.count()} docs).")
        print(f"Retrieving for: '{query_text}'")

        query_params = {
            "query_texts": [query_text],
            "n_results": n_results,
            "include": ['documents', 'distances', 'metadatas'],
        }
        if document_filter:
            print(f"Filtering by document: {document_filter}")
            query_params["where"] = {"source_document": document_filter}

        return collection.query(**query_params)

    except Exception as exc:
        print(f"Retrieval error: {exc}")
        return None

In [48]:
def re_rank_documents(query, retrieved_results, re_ranker_model, n_results=5):
    """
    Re-rank retrieved chunks with a CrossEncoder and return the top
    n_results in the same dict format expected by generate_answer().
    Returns None when retrieved_results is empty.
    """
    if not retrieved_results or not retrieved_results.get('documents', [[]])[0]:
        return None

    docs      = retrieved_results['documents'][0]
    metadatas = retrieved_results['metadatas'][0]

    print("Re-ranking documents...")
    scores = re_ranker_model.predict([[query, doc] for doc in docs])

    # Combine scores, docs, and metadatas, then sort
    combined_results = sorted(
        zip(scores, docs, metadatas),
        key=lambda x: x[0],
        reverse=True,
    )[:n_results]

    print(f"Re-ranked — top {len(combined_results)} selected.")

    # Reconstruct the expected dict format for generate_answer()
    ranked_ids       = []
    ranked_documents = []
    ranked_metadatas = []

    for i, (score, doc, metadata) in enumerate(combined_results):
        ranked_ids.append(f"{metadata.get('source_document', 'unknown')}_page_{metadata.get('page_number', i+1)}")
        ranked_documents.append(doc)
        ranked_metadatas.append(metadata)

    return {
        'ids':       [ranked_ids],
        'documents': [ranked_documents],
        'metadatas': [ranked_metadatas],
    }

In [51]:
def generate_answer(query, retrieved_documents, client=openai_client, model=OPENAI_MODEL):
    """
    Build a prompt from the retrieved context and generate a concise,
    accurate answer using OpenAI gpt-4o-mini.

    Unlike distilgpt2, this model:
      - Follows instructions reliably
      - Stays grounded in the provided context
      - Returns coherent medical summaries

    Returns the answer string.
    """
    context = "\n\n".join(retrieved_documents['documents'][0])

    system_prompt = (
        "You are a helpful medical document assistant. "
        "Answer the user's question using ONLY the provided context. "
        "If the context does not contain enough information to answer, "
        "say so clearly. Be concise and accurate."
    )

    user_prompt = f"""Context from medical documents:
---
{context}
---

Question: {query}"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,    # low temperature = factual, grounded answers
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

print("All RAG functions defined.")

All RAG functions defined.


### 3. Orchestrate RAG Pipeline

Runs the full pipeline in sequence: load → chunk → embed → store in ChromaDB.

In [53]:
# 1. Load PDFs
df_docs = load_and_process_pdfs(DATA_DIRECTORY)
display(df_docs.head())

if df_docs.empty:
    print("No documents processed — aborting pipeline.")
else:
    # 2. Chunk documents
    df_chunks = chunk_documents(df_docs)
    display(df_chunks.head())

    # 3. Generate embeddings (batched — much faster than row-by-row)
    df_chunks_embedded, embedding_model = generate_embeddings(df_chunks, EMBEDDING_MODEL_NAME)
    display(df_chunks_embedded.head())

    # 4. Populate ChromaDB
    chroma_client, medical_collection = setup_chromadb(
        df_chunks_embedded, CHROMADB_PATH, COLLECTION_NAME, EMBEDDING_MODEL_NAME
    )

No text from PyPDF2 for '12Jan2025_thyrocare.pdf' — trying OCR page by page...
  OCR succeeded for '12Jan2025_thyrocare.pdf'. 16 pages extracted.
No text from PyPDF2 for 'Dr_Gopichand_Urologist_yashoda_2-May-2025.pdf' — trying OCR page by page...
  OCR succeeded for 'Dr_Gopichand_Urologist_yashoda_2-May-2025.pdf'. 2 pages extracted.
No text from PyPDF2 for '14-Feb_Yashoda_DiscyargeSummery.pdf' — trying OCR page by page...
  OCR succeeded for '14-Feb_Yashoda_DiscyargeSummery.pdf'. 9 pages extracted.
No text from PyPDF2 for '2-Feb-PusCulture.pdf' — trying OCR page by page...
  OCR succeeded for '2-Feb-PusCulture.pdf'. 2 pages extracted.
PyPDF2 extracted text from '1 pages of '1-May-2025-Pus_Culture.pdf'.
PyPDF2 extracted text from '2 pages of '1-May-2025-UrineCulture.pdf'.
No text from PyPDF2 for '12-Feb-2015-CBP-CUE-RFT-PUSCULTURE-CT-ABDOMEN.pdf' — trying OCR page by page...
  OCR succeeded for '12-Feb-2015-CBP-CUE-RFT-PUSCULTURE-CT-ABDOMEN.pdf'. 8 pages extracted.
No text from PyPDF2 f

,doc_id,page_number,text
0,12Jan2025_thyrocare.pdf,1,—\n|\na\non\nProne\na\nTm\nrd\nimmer\nMmuneaea...
1,12Jan2025_thyrocare.pdf,2,7\n\n \n \n\n4 Thyrocare’\n\nTests you can tru...
2,12Jan2025_thyrocare.pdf,3,B Thyrocare\n\nTests you can trust\n\nPROCESSE...
3,12Jan2025_thyrocare.pdf,4,"PROCESSED AT:\nThyrocare\n\nH. NO. 1-9-645,Vid..."
4,12Jan2025_thyrocare.pdf,5,UAT\n\n \n\n \n\nD160586591\n\nPROCESSED AT...


Treated 89 pages as chunks.


,source_document,page_number,text
0,12Jan2025_thyrocare.pdf,1,—\n|\na\non\nProne\na\nTm\nrd\nimmer\nMmuneaea...
1,12Jan2025_thyrocare.pdf,2,7\n\n \n \n\n4 Thyrocare’\n\nTests you can tru...
2,12Jan2025_thyrocare.pdf,3,B Thyrocare\n\nTests you can trust\n\nPROCESSE...
3,12Jan2025_thyrocare.pdf,4,"PROCESSED AT:\nThyrocare\n\nH. NO. 1-9-645,Vid..."
4,12Jan2025_thyrocare.pdf,5,UAT\n\n \n\n \n\nD160586591\n\nPROCESSED AT...


Initialising embedding model: all-MiniLM-L6-v2...


...

Generating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Done — 89 embeddings, 384 dimensions each.


,source_document,page_number,text,embedding
0,12Jan2025_thyrocare.pdf,1,—\n|\na\non\nProne\na\nTm\nrd\nimmer\nMmuneaea...,"[-0.03129018470644951, -0.036078423261642456, ..."
1,12Jan2025_thyrocare.pdf,2,7\n\n \n \n\n4 Thyrocare’\n\nTests you can tru...,"[-0.04337003454566002, -0.06740792095661163, -..."
2,12Jan2025_thyrocare.pdf,3,B Thyrocare\n\nTests you can trust\n\nPROCESSE...,"[-0.0669061616063118, -0.03269572928547859, -0..."
3,12Jan2025_thyrocare.pdf,4,"PROCESSED AT:\nThyrocare\n\nH. NO. 1-9-645,Vid...","[-0.09220968186855316, -0.03237421065568924, -..."
4,12Jan2025_thyrocare.pdf,5,UAT\n\n \n\n \n\nD160586591\n\nPROCESSED AT...,"[-0.08912836760282516, -0.0718722864985466, -0..."


Initialising ChromaDB at: /content/drive/MyDrive/Upgrad_GenAI/HelpmateAI_dataset/chromadb_data_v2
Deleted existing collection 'medical_documents_rag'.
Adding 89 documents to 'medical_documents_rag'...
Collection populated with 89 documents.


### 4. Verify ChromaDB Collection

In [54]:
client = chromadb.PersistentClient(path=CHROMADB_PATH)
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL_NAME
)
medical_collection = client.get_collection(
    name=COLLECTION_NAME, embedding_function=embed_fn
)
print(f"Documents in '{COLLECTION_NAME}': {medical_collection.count()}")

if os.path.exists(CHROMADB_PATH):
    print("ChromaDB directory contents:", os.listdir(CHROMADB_PATH))
else:
    print("ChromaDB path does not exist yet.")

Documents in 'medical_documents_rag': 89
ChromaDB directory contents: ['af5aef66-6000-4f86-859c-135ad4c09125', '88f71354-19a3-480c-b686-9cbee53121bd', 'chroma.sqlite3', 'f3457610-e6a4-43e1-be45-f44e194598e9']


### 5. Interactive RAG Query Interface

Each query is processed in 4 steps:
1. **Retrieve** — top 10 chunks from ChromaDB (semantic search)
2. **Re-rank** — CrossEncoder narrows to top 3
3. **Generate** — `gpt-4o-mini` produces a grounded answer from the context
4. **Display** — shows sources + final answer

Type `exit` to quit.

In [ ]:
DOC_FILTER_PATTERN = re.compile(
    r'\bsummarize\s+(?:the\s+file\s+)?(\S+\.pdf)\b', re.IGNORECASE
)

while True:
    user_query = input("\nEnter your medical query (or 'exit' to quit): ").strip()

    if user_query.lower() == 'exit':
        print("Exiting RAG session.")
        break

    if not user_query:
        continue

    # Optional: filter to a specific PDF when the query targets one
    doc_filter = None
    match = DOC_FILTER_PATTERN.search(user_query)
    if match:
        doc_filter = match.group(1)
        print(f"Detected specific-document request: {doc_filter}")

    # Step 1 — Retrieve
    initial_results = retrieve_documents(
        user_query, CHROMADB_PATH, COLLECTION_NAME, EMBEDDING_MODEL_NAME,
        n_results=10, document_filter=doc_filter,
    )

    # Step 2 — Re-rank
    rag_results = None
    if initial_results:
        print("\n--- Re-ranking Retrieved Documents ---")
        rag_results = re_rank_documents(
            user_query, initial_results, re_ranker, n_results=5
        )

    # Step 3 — Display retrieved context + generate answer
    print("\n--- Retrieved Documents (for RAG) ---")
    if rag_results and rag_results['ids'][0]:
        for i, (doc, meta) in enumerate(
            zip(rag_results['documents'][0], rag_results['metadatas'][0]), start=1
        ):
            print(f"\n--- Result {i} ---")
            print(f"Source: {meta.get('source_document', 'unknown')}")
            print(f"Content:\n{doc}")

        # Step 4 — Generate answer via OpenAI gpt-4o-mini
        print("\n--- Generating Answer with gpt-4o-mini ---")
        answer = generate_answer(user_query, rag_results)
        print("\n--- Final RAG Answer ---")
        print(answer)
    else:
        print("No relevant documents found or retrieval error.")

    print("\n" + "-" * 70)
    confirm = input("Press Enter for a new query, or type 'exit' to quit: ")
    if confirm.strip().lower() == 'exit':
        print("Exiting RAG session.")
        break


Enter your medical query (or 'exit' to quit): Summerize Dr. GOPICHAND M prescription
Connected to 'medical_documents_rag' (89 docs).
Retrieving for: 'Summerize Dr. GOPICHAND M prescription'

--- Re-ranking Retrieved Documents ---
Re-ranking documents...
Re-ranked — top 5 selected.

--- Retrieved Documents (for RAG) ---

--- Result 1 ---
Source: 14-Feb_Yashoda_DiscyargeSummery.pdf
Content:
< Y Ea Token No: 16
Name : MR. KASI VISWANADHAM SUBUDHI KONCHADA

YASHODA
Slot:11:30 Room NO:243

HOSPITALS Age : 81 Year(s) Gender: Male
YH No : 400617651 Date : 21/03/2025 10:39
City : HYDERABAD Consultation Fee: Rs. 800
Ref By: Rec.No: DFV446414/25 GENERAL
FOLLOW UP VISIT

 

    

    

Dr.SREEKANTH K

M.S , M.Ch.(Surgical Oncology)
Clinical Director

Sr.Consultant Surgical Oncologist
& Robotic Surgeon

Reg No:20692

Mobile: 9849977889

Email: drsreekanthk@gmail.com

Fly ® Ca poate dudytelo_
: DaumovectaX olascong compra
© and eorkon a rnol Canal

BH COmwanine Mn

Ne cfrean epinvcloc “b fonew nod